# 🎙️ Whisper Subtitle Generator — Google Colab (GPU)

**~3–5 mins per 30-min video** using free T4 GPU

### Before running:
1. Go to **Runtime → Change runtime type → T4 GPU** → Save
2. Upload your videos to **Google Drive** (any folder)
3. Fill in the config cell below
4. Run all cells top to bottom (**Runtime → Run all**)

---

## Step 1 — Install Whisper

In [ ]:
# Install faster-whisper (uses GPU automatically, much faster than openai-whisper)
!pip install faster-whisper -q
print('✅ faster-whisper installed')

## Step 2 — Mount Google Drive
A popup will ask you to sign in and allow access. Click **Connect to Google Drive**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted at /content/drive/My Drive/')

## Step 3 — Configuration
**Edit these settings before running the next cells.**

Your Drive path:
- If your videos are in `My Drive/GOTO_CNTNT/1. Akhiri Moujza/` → set `VIDEO_FOLDER` to that path below
- All subfolders are scanned automatically

In [ ]:
# ─────────────────────────────────────────────────────────
#  EDIT THESE SETTINGS
# ─────────────────────────────────────────────────────────

# Your video folder inside Google Drive
# Format: /content/drive/My Drive/YOUR_FOLDER_NAME
VIDEO_FOLDER = "/content/drive/My Drive/GOTO_CNTNT/1. Akhiri Moujza"

# Model size:
#   tiny   → fastest, lowest accuracy
#   base   → fast, decent
#   small  → good balance
#   medium → best balance
#   large-v3 → highest accuracy ← RECOMMENDED on Colab GPU
MODEL = "large-v3"

# Language: use ISO code or "auto" to detect automatically
# Examples: en, ur, ar, hi, fr, auto
LANGUAGE = "ur"

# Output format: "srt", "vtt", or "both"
OUTPUT_FORMAT = "srt"

# Skip videos that already have subtitle files (True/False)
SKIP_EXISTING = True

# Optional: HuggingFace token for faster model download
# Get free token at https://huggingface.co/settings/tokens
HF_TOKEN = ""  # leave empty if you don't have one

# ─────────────────────────────────────────────────────────
print(f"📁 Video folder : {VIDEO_FOLDER}")
print(f"🤖 Model        : {MODEL}")
print(f"🌐 Language     : {LANGUAGE}")
print(f"📄 Output format: {OUTPUT_FORMAT}")
print(f"⏭️  Skip existing: {SKIP_EXISTING}")

## Step 4 — Check GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    print(f"✅ GPU detected: {gpu}")
    DEVICE = "cuda"
    COMPUTE_TYPE = "float16"
else:
    print("⚠️  No GPU found — running on CPU (slow)")
    print("   Go to Runtime → Change runtime type → T4 GPU → Save")
    DEVICE = "cpu"
    COMPUTE_TYPE = "int8"

## Step 5 — Run Transcription
This will process all videos in your folder and save `.srt` files next to each video in Google Drive.

In [ ]:
import os
import glob
import time

# Set HuggingFace token if provided
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("🔑 HuggingFace token set")

from faster_whisper import WhisperModel

# ── Helpers ───────────────────────────────────────────────────────────────────
VIDEO_EXTENSIONS = ("*.mp4", "*.mkv", "*.avi", "*.mov", "*.wmv", "*.webm", "*.flv", "*.m4v")

def format_ts_srt(seconds):
    h  = int(seconds // 3600)
    m  = int((seconds % 3600) // 60)
    s  = int(seconds % 60)
    ms = int((seconds % 1) * 1000)
    return f"{h:02}:{m:02}:{s:02},{ms:03}"

def format_ts_vtt(seconds):
    h  = int(seconds // 3600)
    m  = int((seconds % 3600) // 60)
    s  = int(seconds % 60)
    ms = int((seconds % 1) * 1000)
    return f"{h:02}:{m:02}:{s:02}.{ms:03}"

def segments_to_srt(segments):
    lines = []
    for i, seg in enumerate(segments, 1):
        lines.append(f"{i}\n{format_ts_srt(seg['start'])} --> {format_ts_srt(seg['end'])}\n{seg['text'].strip()}\n")
    return "\n".join(lines)

def segments_to_vtt(segments):
    lines = ["WEBVTT\n"]
    for i, seg in enumerate(segments, 1):
        lines.append(f"{i}\n{format_ts_vtt(seg['start'])} --> {format_ts_vtt(seg['end'])}\n{seg['text'].strip()}\n")
    return "\n".join(lines)

def fmt_time(s):
    return f"{int(s//60)}m {int(s%60)}s" if s >= 60 else f"{s:.1f}s"

# ── Validate folder ───────────────────────────────────────────────────────────
if not os.path.isdir(VIDEO_FOLDER):
    raise FileNotFoundError(
        f"\n❌ Folder not found: {VIDEO_FOLDER}\n"
        "Check that:\n"
        "  1. Google Drive is mounted (Step 2)\n"
        "  2. The folder path in Step 3 is correct\n"
        "  3. You uploaded your videos to Google Drive"
    )

# ── Gather videos ─────────────────────────────────────────────────────────────
videos = []
for ext in VIDEO_EXTENSIONS:
    videos.extend(glob.glob(os.path.join(VIDEO_FOLDER, "**", ext), recursive=True))
videos.sort()

if not videos:
    raise FileNotFoundError(
        f"\n❌ No video files found in: {VIDEO_FOLDER}\n"
        "Supported formats: mp4, mkv, avi, mov, wmv, webm, flv, m4v"
    )

print(f"\n🎬 Found {len(videos)} video(s) to process\n")

# ── Load model ────────────────────────────────────────────────────────────────
print(f"⏳ Loading '{MODEL}' model on {DEVICE.upper()}...")
print("   (First run downloads the model — one-time only)\n")
model = WhisperModel(MODEL, device=DEVICE, compute_type=COMPUTE_TYPE)
print(f"✅ Model ready\n")
print("─" * 60)

# ── Process videos ────────────────────────────────────────────────────────────
lang = None if LANGUAGE.lower() == "auto" else LANGUAGE
total_start   = time.time()
success_count = 0
skip_count    = 0
error_count   = 0

for idx, video_path in enumerate(videos, 1):
    filename  = os.path.basename(video_path)
    base_path = os.path.splitext(video_path)[0]
    srt_path  = base_path + ".srt"
    vtt_path  = base_path + ".vtt"
    prefix    = f"[{idx:>3}/{len(videos)}]"

    # Check skip
    already_done = (
        (OUTPUT_FORMAT in ("srt", "both") and os.path.exists(srt_path)) or
        (OUTPUT_FORMAT in ("vtt", "both") and os.path.exists(vtt_path))
    )
    if SKIP_EXISTING and already_done:
        print(f"  {prefix} ⏭️  SKIP   {filename}")
        skip_count += 1
        continue

    print(f"  {prefix} ⏳ ...   {filename}")
    step_start = time.time()

    try:
        seg_gen, info = model.transcribe(video_path, language=lang, task="transcribe")
        segments = []
        for s in seg_gen:
            segments.append({"start": s.start, "end": s.end, "text": s.text})
            pct = min(100.0, (s.end / info.duration) * 100) if info.duration > 0 else 0
            print(f"\r         ↳ Progress: {pct:4.1f}%  ({s.end:.1f}s / {info.duration:.1f}s)    ", end="", flush=True)
        print("\r" + " " * 65 + "\r", end="", flush=True)

        if OUTPUT_FORMAT in ("srt", "both"):
            with open(srt_path, "w", encoding="utf-8") as f:
                f.write(segments_to_srt(segments))

        if OUTPUT_FORMAT in ("vtt", "both"):
            with open(vtt_path, "w", encoding="utf-8") as f:
                f.write(segments_to_vtt(segments))

        elapsed = time.time() - step_start
        print(f"  {prefix} ✅ OK    {filename}  ({fmt_time(elapsed)})")
        success_count += 1

    except Exception as e:
        print(f"  {prefix} ❌ ERR   {filename}")
        print(f"           {e}")
        error_count += 1

# ── Summary ───────────────────────────────────────────────────────────────────
total_elapsed = time.time() - total_start
print()
print("─" * 60)
print(f"  ✅ Generated : {success_count} subtitle file(s)")
if skip_count:  print(f"  ⏭️  Skipped  : {skip_count} (already existed)")
if error_count: print(f"  ❌ Errors   : {error_count}")
print(f"  ⏱️  Total time: {fmt_time(total_elapsed)}")
print("─" * 60)
print()
print("🎉 Done! Your .srt files are saved next to your videos in Google Drive.")
print("   You can download them directly from drive.google.com")